In [ ]:
import base64
from pathlib import Path
root = Path('/content/hsc_im_atac_lsi')
root.mkdir(parents=True, exist_ok=True)
rds_path = root / 'Augur_HSCIvsHSCII_ATAC_lsi.rds'
encoded_rds = (
    '...'
    'AMwkS410nDsA'
)
rds_path.write_bytes(base64.b64decode(encoded_rds))
del encoded_rds
print('RDS restored:', round(rds_path.stat().st_size / 1e6, 2), 'MB')

In [ ]:
import subprocess, sys, shutil
if not shutil.which('Rscript'):
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','r-base-core'],check=True)
check_matrix = subprocess.run(['Rscript','-e','quit(status=if(requireNamespace("Matrix",quietly=TRUE)) 0 else 1)'])
if check_matrix.returncode:
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','r-cran-matrix'],check=True)
subprocess.run([sys.executable,'-m','pip','-q','install','scanpy','igraph','leidenalg'],check=True)
print('Python and R environments ready.')

In [ ]:
from pathlib import Path
r_script = root / 'convert_Augur_ATAC_LSI.R'
r_script.write_text("# Convert only the published ATAC-LSI feature matrix and cell labels.\n# No Seurat/Signac installation required.\nsuppressPackageStartupMessages(library(Matrix))\nd <- readRDS('/content/hsc_im_atac_lsi/Augur_HSCIvsHSCII_ATAC_lsi.rds')\ncat('Augur object fields:', paste(names(d), collapse=', '), '\\n')\nstopifnot(is.list(d), all(c('X','y') %in% names(d)))\nX <- d$X\ny <- as.character(d$y)\ncat('Original ATAC matrix dimensions:', paste(dim(X), collapse=' x '), '\\n')\ncat('Published labels:', paste(names(table(y)), as.integer(table(y)), collapse='; '), '\\n')\nif (ncol(X) == length(y)) {\n   z <- t(X)\n} else if (nrow(X) == length(y)) {\n   z <- X\n} else {\n   stop('Input X dimensions do not match the length of y; inspect Augur object before proceeding.')\n}\nz <- as.matrix(z)\nif (nrow(z) != length(y)) stop('Cell-label length mismatch')\nif (ncol(z) < 2) stop('Expected multiple LSI features')\nif (any(!is.finite(z))) stop('Non-finite LSI feature values detected')\nids <- rownames(z)\nif (is.null(ids) || any(!nzchar(ids)) || anyDuplicated(ids)) {\n    ids <- paste0('Cell_',seq_len(nrow(z)))\n    cat('Warning: input lacks unique cell identifiers; using generated row IDs.\\n')\n}\nwrite.csv(data.frame(Cell=ids,published_type=y, check.names=FALSE), '/content/hsc_im_atac_lsi/ATAC_cell_labels.csv', row.names=FALSE)\nwrite.csv(z, '/content/hsc_im_atac_lsi/ATAC_LSI_features.csv', row.names=FALSE)\ncat('Converted:', nrow(z),'cells and',ncol(z),'published ATAC features.\\n')\n")
result = subprocess.run(['Rscript',str(r_script)],capture_output=True,text=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError('R data conversion failed. See the output above.')

In [ ]:
import numpy as np, pandas as pd, scanpy as sc, matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

labels = pd.read_csv(root / 'ATAC_cell_labels.csv')
features = pd.read_csv(root / 'ATAC_LSI_features.csv').to_numpy(dtype='float32')
assert len(labels) == len(features)
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=110, facecolor='white')

# New graph/clustering/UMAP calculated from authors' ATAC LSI representation.
a = sc.AnnData(X=features)
a.obs_names = labels['Cell'].astype(str).to_numpy()
a.obs['published_type'] = labels['published_type'].astype(str).to_numpy()
sc.pp.neighbors(a, n_neighbors=20, use_rep='X', random_state=123)
sc.tl.leiden(a, resolution=0.5, key_added='atac_leiden', flavor='igraph', directed=False, n_iterations=2)
sc.tl.umap(a, random_state=123)

out = root / 'results'
out.mkdir(exist_ok=True)
fig, axs = plt.subplots(1, 2, figsize=(13,5))
sc.pl.umap(a, color='atac_leiden', ax=axs[0], show=False, size=7, frameon=False, title='New ATAC-LSI Leiden clusters')
sc.pl.umap(a, color='published_type', ax=axs[1], show=False, size=7, frameon=False, title='Published HSC labels')
fig.tight_layout()
fig.savefig(out / 'ATAC_LSI_reclustering.png', dpi=250, bbox_inches='tight')
plt.show()
plt.close(fig)

comparison = pd.crosstab(a.obs['atac_leiden'], a.obs['published_type'])
comparison.to_csv(out / 'ATAC_clusters_vs_published_HSC_labels.csv')
coords = a.obs[['published_type','atac_leiden']].copy()
coords['ATAC_UMAP1'] = a.obsm['X_umap'][:,0]
coords['ATAC_UMAP2'] = a.obsm['X_umap'][:,1]
coords.to_csv(out / 'ATAC_LSI_UMAP_cell_annotations.csv',index_label='Cell')

metrics = pd.DataFrame([{'n_cells':len(a),'n_lsi_features':a.n_vars,'n_leiden_clusters':a.obs['atac_leiden'].nunique(),
    'ARI_vs_published_HSC_labels':adjusted_rand_score(a.obs['published_type'],a.obs['atac_leiden']),
    'NMI_vs_published_HSC_labels':normalized_mutual_info_score(a.obs['published_type'],a.obs['atac_leiden'])}])
metrics.to_csv(out / 'ATAC_LSI_analysis_metrics.csv',index=False)

readme = '# HSC-iM xenograft ATAC LSI reclustering\n\nSource: Augur_HSCIvsHSCII_ATAC_lsi.rds (Zeng, Nagree et al., Nature 2026; Zenodo 18850871).\n\nThe original ATAC LSI input is AUTHOR-COMPUTED. The neighbor graph, Leiden clusters (resolution 0.5), and UMAP (seed 123) are NEWLY computed from that representation. Comparison labels are author annotations. This is NOT independent ATAC peak-matrix preprocessing or Seurat WNN multimodal integration. No donor-level statistical tests are carried out.\n'
(out / 'README.md').write_text(readme)

zip_path = Path('/content/HSC_iM_ATAC_LSI_Clustering_Results.zip')
with ZipFile(zip_path,'w',ZIP_DEFLATED) as z:
    for file in sorted(out.iterdir()):
        if file.is_file():z.write(file,file.name)
print('Output files:',[p.name for p in out.iterdir()])
print('Cells:',len(a),'Clusters:',a.obs['atac_leiden'].nunique())
print(comparison.to_string())
files.download(str(zip_path))